# Clase 10 - MCP con `stdio` y Streamable HTTP en Colab/local

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/></a>

> Este laboratorio está diseñado para ejecutarse **por defecto en Google Colab**. También funciona en un notebook local con Python/Jupyter.

**Objetivos.** Al terminar deberías poder:

- Preparar un entorno MCP portable para Colab y ejecución local.
- Crear una base SQLite pequeña de ejemplo.
- Implementar un servidor MCP con `FastMCP`.
- Exponer `resources`, `tools` y `prompts`.
- Probar el servidor con cliente MCP por `stdio`.
- Levantar el mismo servidor con transporte `streamable-http` en `localhost`.
- Comparar cuándo conviene `stdio` y cuándo conviene Streamable HTTP.
- Revisar seguridad local: confirmación humana, logs, puertos y separación entre datos y acciones.

El flujo completo ocurre dentro del notebook. No requiere SSH, clientes de escritorio ni APIs externas.


## Arquitectura

El mismo servidor MCP se probará con dos métodos de transporte vistos en clases.

### Parte A: `stdio`

```text
Google Colab o Jupyter local
          |
          | cliente MCP en Python
          | stdio + JSON-RPC
          v
Servidor MCP curso-mcp como subproceso
          |
          | sqlite3 local
          v
Base curso_mcp.db
```

### Parte B: Streamable HTTP

```text
Google Colab o Jupyter local
          |
          | cliente MCP en Python
          | HTTP local: http://127.0.0.1:PUERTO/mcp
          v
Servidor MCP curso-mcp escuchando en localhost
          |
          | sqlite3 local
          v
Base curso_mcp.db
```

`stdio` es ideal para ejecución local simple. Streamable HTTP es útil cuando el servidor debe aceptar conexiones vía red, manejar más de un cliente o integrarse como servicio.


## Paso 0: compatibilidad

| Entorno | `stdio` | Streamable HTTP local |
|---|---:|---:|
| Google Colab | sí | sí, dentro del runtime |
| Jupyter local en Linux/macOS/Windows | sí | sí, en `127.0.0.1` |

En este laboratorio Streamable HTTP se ejecuta solo en `localhost`. Para exponerlo fuera del entorno tendrías que agregar autenticación, TLS, control de origen, rate limiting y políticas de permisos.


In [1]:
# Instalación recomendada para Colab o entorno local limpio.
%pip install -q -U "mcp[cli]" pandas==2.2.2 pydantic nest_asyncio

In [2]:
import json
import os
import platform
import socket
import sqlite3
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("Ejecutable Python:", sys.executable)

Python: 3.12.13
Sistema: Linux-6.6.122+-x86_64-with-glibc2.35
Ejecutable Python: /usr/bin/python3


## Paso 1: crear carpeta de trabajo

In [3]:
if Path("/content").exists():
    LAB_DIR = Path("/content/mcp_lab")
    ENTORNO = "Google Colab"
else:
    LAB_DIR = Path.cwd() / "mcp_lab"
    ENTORNO = "Jupyter/local"

LAB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = LAB_DIR / "curso_mcp.db"
SERVER_PATH = LAB_DIR / "servidor_curso_mcp.py"

print("Entorno detectado:", ENTORNO)
print("LAB_DIR:", LAB_DIR)
print("DB_PATH:", DB_PATH)
print("SERVER_PATH:", SERVER_PATH)


Entorno detectado: Google Colab
LAB_DIR: /content/mcp_lab
DB_PATH: /content/mcp_lab/curso_mcp.db
SERVER_PATH: /content/mcp_lab/servidor_curso_mcp.py


## Paso 2: crear base SQLite demo

La base simula datos del curso. Luego el servidor MCP la expondrá como contexto y herramientas.

In [4]:
def inicializar_sqlite(db_path: Path):
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
    CREATE TABLE clases (
        numero INTEGER PRIMARY KEY,
        titulo TEXT NOT NULL,
        tema TEXT NOT NULL,
        resumen TEXT NOT NULL,
        duracion_min INTEGER NOT NULL
    )
    """)
    cur.execute("""
    CREATE TABLE tareas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        clase_numero INTEGER NOT NULL,
        titulo TEXT NOT NULL,
        prioridad TEXT NOT NULL,
        estado TEXT NOT NULL,
        FOREIGN KEY(clase_numero) REFERENCES clases(numero)
    )
    """)
    clases = [
        (1, "Cuantización, PEFT y despliegue", "LLMs eficientes", "Reduce memoria, adapta modelos con LoRA/QLoRA y despliega de forma responsable.", 120),
        (2, "Aprendizaje por contexto", "Prompting avanzado", "Diseña prompts zero-shot, few-shot, CoT, autoconsistencia y salida estructurada.", 120),
        (3, "Agentes", "Agentes con herramientas", "Construye agentes que perciben, planifican, actúan y registran memoria.", 120),
        (4, "Model Context Protocol", "Integración estándar", "Conecta hosts LLM con tools, resources y prompts mediante MCP.", 120),
    ]
    cur.executemany("INSERT INTO clases VALUES (?, ?, ?, ?, ?)", clases)
    tareas = [
        (1, "Probar LoRA con modelo pequeño", "media", "pendiente"),
        (2, "Comparar zero-shot vs few-shot", "alta", "completada"),
        (3, "Conectar agente a Firestore simulado", "alta", "pendiente"),
        (4, "Crear servidor MCP por stdio", "critica", "pendiente"),
    ]
    cur.executemany("INSERT INTO tareas (clase_numero, titulo, prioridad, estado) VALUES (?, ?, ?, ?)", tareas)
    con.commit()
    con.close()

inicializar_sqlite(DB_PATH)
print("Base creada:", DB_PATH)

Base creada: /content/mcp_lab/curso_mcp.db


In [ ]:
with sqlite3.connect(DB_PATH) as con:
    display(pd.read_sql_query("SELECT * FROM clases", con))
    display(pd.read_sql_query("SELECT * FROM tareas", con))

## Paso 3: contrato del servidor MCP

| Tipo | Nombre/URI | Uso |
|---|---|---|
| Resource | `curso://resumen` | Resumen completo del curso |
| Resource | `clase://{numero}` | Detalle de una clase |
| Resource | `schema://sqlite` | Esquema de la base |
| Tool | `buscar_clases` | Buscar por texto |
| Tool | `obtener_clase` | Obtener clase estructurada |
| Tool | `listar_tareas` | Listar tareas filtradas |
| Tool | `crear_tarea` | Crear tarea con confirmación |
| Tool | `marcar_tarea` | Cambiar estado con confirmación |
| Tool | `resumen_progreso` | Métricas simples |
| Prompt | `preparar_estudio` | Guía de estudio |
| Prompt | `generar_quiz` | Plantilla de quiz |

## Paso 4: escribir servidor MCP

El servidor usa `FastMCP`. El mismo archivo puede arrancar en dos modos:

- `stdio`: comunicación por entrada/salida estándar, sin abrir puertos.
- `streamable-http`: servidor HTTP local en `/mcp`.

En modo `stdio`, `stdout` queda reservado para JSON-RPC; los logs deben ir a `stderr`. En modo HTTP, el servidor escucha en `127.0.0.1` para mantenerlo local al runtime de Colab/Jupyter.


In [ ]:
server_code = '\nimport argparse\nimport json\nimport os\nimport sqlite3\nimport sys\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nfrom mcp.server.fastmcp import FastMCP\nfrom mcp.server.fastmcp.exceptions import ToolError\n\nDB_PATH = Path(os.environ.get("CURSO_MCP_DB", "curso_mcp.db")).resolve()\nmcp = FastMCP("curso-mcp", stateless_http=True, json_response=True)\n\ndef conectar():\n    if not DB_PATH.exists():\n        raise ToolError(f"No existe la base SQLite: {DB_PATH}")\n    con = sqlite3.connect(DB_PATH)\n    con.row_factory = sqlite3.Row\n    return con\n\ndef filas_dict(cursor):\n    return [dict(row) for row in cursor.fetchall()]\n\ndef validar_prioridad(prioridad: str) -> str:\n    prioridad = prioridad.lower().strip()\n    validas = {"baja", "media", "alta", "critica"}\n    if prioridad not in validas:\n        raise ToolError(f"Prioridad inválida: {prioridad}. Usa una de: {sorted(validas)}")\n    return prioridad\n\ndef validar_estado(estado: str) -> str:\n    estado = estado.lower().strip()\n    validos = {"pendiente", "en_progreso", "completada"}\n    if estado not in validos:\n        raise ToolError(f"Estado inválido: {estado}. Usa uno de: {sorted(validos)}")\n    return estado\n\n@mcp.resource("curso://resumen", mime_type="application/json")\ndef recurso_resumen_curso() -> str:\n    """Devuelve un resumen JSON de clases y tareas del curso."""\n    with conectar() as con:\n        clases = filas_dict(con.execute("SELECT * FROM clases ORDER BY numero"))\n        tareas = filas_dict(con.execute("SELECT * FROM tareas ORDER BY id"))\n    return json.dumps({"clases": clases, "tareas": tareas}, ensure_ascii=False, indent=2)\n\n@mcp.resource("clase://{numero}", mime_type="application/json")\ndef recurso_clase(numero: str) -> str:\n    """Devuelve el detalle JSON de una clase por número."""\n    try:\n        numero_int = int(numero)\n    except ValueError:\n        raise ToolError("El número de clase debe ser entero.")\n    with conectar() as con:\n        row = con.execute("SELECT * FROM clases WHERE numero = ?", (numero_int,)).fetchone()\n        if not row:\n            raise ToolError(f"No existe la clase {numero_int}.")\n        tareas = filas_dict(con.execute("SELECT * FROM tareas WHERE clase_numero = ? ORDER BY id", (numero_int,)))\n    return json.dumps({"clase": dict(row), "tareas": tareas}, ensure_ascii=False, indent=2)\n\n@mcp.resource("schema://sqlite", mime_type="text/plain")\ndef recurso_schema() -> str:\n    """Devuelve el esquema SQL disponible para el servidor."""\n    with conectar() as con:\n        rows = con.execute("SELECT name, sql FROM sqlite_master WHERE type=\'table\' ORDER BY name").fetchall()\n    return "\\n\\n".join([f"-- {r[\'name\']}\\n{r[\'sql\']}" for r in rows])\n\n@mcp.tool()\ndef buscar_clases(termino: str) -> list[dict[str, Any]]:\n    """Busca clases por título, tema o resumen."""\n    termino_like = f"%{termino.lower().strip()}%"\n    with conectar() as con:\n        cur = con.execute("""\n            SELECT * FROM clases\n            WHERE lower(titulo) LIKE ? OR lower(tema) LIKE ? OR lower(resumen) LIKE ?\n            ORDER BY numero\n        """, (termino_like, termino_like, termino_like))\n        return filas_dict(cur)\n\n@mcp.tool()\ndef obtener_clase(numero: int) -> dict[str, Any]:\n    """Obtiene una clase y sus tareas asociadas."""\n    with conectar() as con:\n        row = con.execute("SELECT * FROM clases WHERE numero = ?", (numero,)).fetchone()\n        if not row:\n            raise ToolError(f"No existe la clase {numero}.")\n        tareas = filas_dict(con.execute("SELECT * FROM tareas WHERE clase_numero = ? ORDER BY id", (numero,)))\n    return {"clase": dict(row), "tareas": tareas}\n\n@mcp.tool()\ndef listar_tareas(estado: Optional[str] = None, prioridad: Optional[str] = None) -> list[dict[str, Any]]:\n    """Lista tareas opcionalmente filtradas por estado y prioridad."""\n    filtros = []\n    params: list[Any] = []\n    if estado:\n        filtros.append("estado = ?")\n        params.append(validar_estado(estado))\n    if prioridad:\n        filtros.append("prioridad = ?")\n        params.append(validar_prioridad(prioridad))\n    where = " WHERE " + " AND ".join(filtros) if filtros else ""\n    with conectar() as con:\n        cur = con.execute(f"SELECT * FROM tareas{where} ORDER BY id", params)\n        return filas_dict(cur)\n\n@mcp.tool()\ndef crear_tarea(clase_numero: int, titulo: str, prioridad: str = "media", confirmado: bool = False) -> dict[str, Any]:\n    """Crea una tarea. Requiere confirmado=True porque escribe en la base."""\n    if not confirmado:\n        raise ToolError("Crear tareas modifica la base. Reintenta con confirmado=True si el usuario lo aprueba.")\n    prioridad = validar_prioridad(prioridad)\n    titulo = titulo.strip()\n    if not titulo:\n        raise ToolError("El título no puede estar vacío.")\n    with conectar() as con:\n        clase = con.execute("SELECT numero FROM clases WHERE numero = ?", (clase_numero,)).fetchone()\n        if not clase:\n            raise ToolError(f"No existe la clase {clase_numero}.")\n        cur = con.execute("INSERT INTO tareas (clase_numero, titulo, prioridad, estado) VALUES (?, ?, ?, \'pendiente\')", (clase_numero, titulo, prioridad))\n        con.commit()\n        row = con.execute("SELECT * FROM tareas WHERE id = ?", (cur.lastrowid,)).fetchone()\n        return dict(row)\n\n@mcp.tool()\ndef marcar_tarea(tarea_id: int, estado: str, confirmado: bool = False) -> dict[str, Any]:\n    """Cambia el estado de una tarea. Requiere confirmado=True."""\n    if not confirmado:\n        raise ToolError("Cambiar estado modifica la base. Reintenta con confirmado=True si el usuario lo aprueba.")\n    estado = validar_estado(estado)\n    with conectar() as con:\n        row = con.execute("SELECT * FROM tareas WHERE id = ?", (tarea_id,)).fetchone()\n        if not row:\n            raise ToolError(f"No existe la tarea {tarea_id}.")\n        con.execute("UPDATE tareas SET estado = ? WHERE id = ?", (estado, tarea_id))\n        con.commit()\n        row = con.execute("SELECT * FROM tareas WHERE id = ?", (tarea_id,)).fetchone()\n        return dict(row)\n\n@mcp.tool()\ndef resumen_progreso() -> dict[str, Any]:\n    """Devuelve métricas de avance del curso y tareas."""\n    with conectar() as con:\n        total_clases = con.execute("SELECT COUNT(*) AS n FROM clases").fetchone()["n"]\n        total_tareas = con.execute("SELECT COUNT(*) AS n FROM tareas").fetchone()["n"]\n        por_estado = filas_dict(con.execute("SELECT estado, COUNT(*) AS cantidad FROM tareas GROUP BY estado ORDER BY estado"))\n        criticas = filas_dict(con.execute("SELECT * FROM tareas WHERE prioridad=\'critica\' AND estado!=\'completada\' ORDER BY id"))\n    return {"total_clases": total_clases, "total_tareas": total_tareas, "tareas_por_estado": por_estado, "tareas_criticas_pendientes": criticas}\n\n@mcp.prompt(title="Preparar estudio")\ndef preparar_estudio(numero_clase: int) -> str:\n    """Genera una plantilla para estudiar una clase."""\n    return f"""Prepara una guía de estudio para la clase {numero_clase}.\nUsa el recurso clase://{numero_clase} como contexto.\nIncluye objetivos, conceptos clave, errores frecuentes, preguntas de práctica y tareas recomendadas."""\n\n@mcp.prompt(title="Generar quiz")\ndef generar_quiz(tema: str, dificultad: str = "media") -> str:\n    """Genera una plantilla para crear un quiz del curso."""\n    return f"""Crea un quiz en español sobre {tema} con dificultad {dificultad}.\nIncluye 5 preguntas de selección múltiple, 2 preguntas cortas y respuestas al final."""\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Servidor MCP del curso")\n    parser.add_argument("--transport", choices=["stdio", "streamable-http"], default=os.environ.get("CURSO_MCP_TRANSPORT", "stdio"))\n    parser.add_argument("--host", default=os.environ.get("CURSO_MCP_HOST", "127.0.0.1"))\n    parser.add_argument("--port", type=int, default=int(os.environ.get("CURSO_MCP_PORT", "8000")))\n    args = parser.parse_args()\n\n    print(f"curso-mcp usando DB: {DB_PATH}", file=sys.stderr)\n    if args.transport == "stdio":\n        mcp.run(transport="stdio")\n    else:\n        print(f"curso-mcp HTTP en http://{args.host}:{args.port}/mcp", file=sys.stderr)\n        mcp.run(transport="streamable-http", host=args.host, port=args.port)\n'

SERVER_PATH.write_text(server_code, encoding="utf-8")
print("Servidor escrito en:", SERVER_PATH)
print(SERVER_PATH.read_text(encoding="utf-8")[:1200])


## Paso 5: compilar servidor

In [ ]:
import py_compile
py_compile.compile(str(SERVER_PATH), doraise=True)
print("Servidor compila correctamente.")

## Paso 6: crear cliente MCP por `stdio`

In [ ]:
import asyncio
import nest_asyncio
from pydantic import AnyUrl
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_PATH)],
    env={"CURSO_MCP_DB": str(DB_PATH)},
)
print(server_params)

## Paso 7: listar tools, resources y prompts

In [ ]:
async def inspeccionar_servidor():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            resources = await session.list_resources()
            templates = await session.list_resource_templates()
            prompts = await session.list_prompts()
            return {
                "tools": [(t.name, t.description) for t in tools.tools],
                "resources": [(str(r.uri), r.name, r.mimeType) for r in resources.resources],
                "resource_templates": [(rt.uriTemplate, rt.name) for rt in templates.resourceTemplates],
                "prompts": [(p.name, p.description) for p in prompts.prompts],
            }

info_mcp = asyncio.get_event_loop().run_until_complete(inspeccionar_servidor())
print(json.dumps(info_mcp, ensure_ascii=False, indent=2))

## Paso 8: leer resources

Los resources son contexto controlado por la aplicación y no deberían tener efectos secundarios.

In [ ]:
async def leer_recursos_demo():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            resumen = await session.read_resource(AnyUrl("curso://resumen"))
            clase = await session.read_resource(AnyUrl("clase://4"))
            schema = await session.read_resource(AnyUrl("schema://sqlite"))
            return resumen, clase, schema

resumen, clase, schema = asyncio.get_event_loop().run_until_complete(leer_recursos_demo())
for nombre, recurso in [("curso://resumen", resumen), ("clase://4", clase), ("schema://sqlite", schema)]:
    print("=" * 80)
    print(nombre)
    contenido = recurso.contents[0]
    print(contenido.text[:1500] if hasattr(contenido, "text") else contenido)

## Paso 9: llamar tools de lectura

In [ ]:
async def llamar_tools_lectura():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            busqueda = await session.call_tool("buscar_clases", {"termino": "agentes"})
            clase = await session.call_tool("obtener_clase", {"numero": 3})
            tareas = await session.call_tool("listar_tareas", {"estado": "pendiente"})
            progreso = await session.call_tool("resumen_progreso", {})
            return busqueda, clase, tareas, progreso

resultados = asyncio.get_event_loop().run_until_complete(llamar_tools_lectura())
for nombre, resultado in zip(["buscar_clases", "obtener_clase", "listar_tareas", "resumen_progreso"], resultados):
    print("=" * 80)
    print(nombre)
    print("isError:", getattr(resultado, "isError", False))
    print(json.dumps(getattr(resultado, "structuredContent", None), ensure_ascii=False, indent=2)[:2000])

## Paso 10: tools con efectos secundarios y aprobación

`crear_tarea` y `marcar_tarea` modifican la base. El servidor exige `confirmado=True`.

In [ ]:
async def probar_sin_confirmar():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.call_tool("crear_tarea", {
                "clase_numero": 4,
                "titulo": "Documentar ejecución MCP en Colab/local",
                "prioridad": "alta",
                "confirmado": False,
            })

sin_confirmar = asyncio.get_event_loop().run_until_complete(probar_sin_confirmar())
print("isError:", sin_confirmar.isError)
for c in sin_confirmar.content:
    if isinstance(c, types.TextContent):
        print(c.text)

In [ ]:
async def crear_y_marcar_confirmada():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            creada = await session.call_tool("crear_tarea", {
                "clase_numero": 4,
                "titulo": "Documentar ejecución MCP en Colab/local",
                "prioridad": "alta",
                "confirmado": True,
            })
            nueva_id = creada.structuredContent["id"]
            marcada = await session.call_tool("marcar_tarea", {
                "tarea_id": nueva_id,
                "estado": "en_progreso",
                "confirmado": True,
            })
            progreso = await session.call_tool("resumen_progreso", {})
            return creada, marcada, progreso

creada, marcada, progreso = asyncio.get_event_loop().run_until_complete(crear_y_marcar_confirmada())
print("Tarea creada:")
print(json.dumps(creada.structuredContent, ensure_ascii=False, indent=2))
print("\nTarea marcada:")
print(json.dumps(marcada.structuredContent, ensure_ascii=False, indent=2))
print("\nProgreso:")
print(json.dumps(progreso.structuredContent, ensure_ascii=False, indent=2))


## Paso 11: obtener prompts reutilizables

In [ ]:
async def obtener_prompts_demo():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            p1 = await session.get_prompt("preparar_estudio", arguments={"numero_clase": "4"})
            p2 = await session.get_prompt("generar_quiz", arguments={"tema": "MCP", "dificultad": "media"})
            return p1, p2

p1, p2 = asyncio.get_event_loop().run_until_complete(obtener_prompts_demo())
for nombre, prompt in [("preparar_estudio", p1), ("generar_quiz", p2)]:
    print("=" * 80)
    print(nombre)
    for msg in prompt.messages:
        print(msg.role, "->", msg.content)

## Paso 12: simular flujo host-agente

Un host real decidiría con un LLM. Aquí hacemos una política simple para mostrar el patrón.

In [ ]:
async def asistente_simulado(pregunta: str):
    pregunta_l = pregunta.lower()
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if "pendiente" in pregunta_l or "tarea" in pregunta_l:
                resultado = await session.call_tool("listar_tareas", {"estado": "pendiente"})
                datos = resultado.structuredContent or []
                lineas = [f"- {d['titulo']} ({d['prioridad']})" for d in datos]
                return f"Encontré {len(datos)} tareas pendientes:\n" + "\n".join(lineas)
            if "mcp" in pregunta_l or "clase 4" in pregunta_l:
                recurso = await session.read_resource(AnyUrl("clase://4"))
                return "Contexto recuperado de clase://4:\n" + recurso.contents[0].text[:1200]
            resultado = await session.call_tool("resumen_progreso", {})
            return "Resumen de progreso:\n" + json.dumps(resultado.structuredContent, ensure_ascii=False, indent=2)

for pregunta in ["¿Qué tareas pendientes tengo?", "Dame contexto de la clase 4 de MCP", "¿Cómo va el progreso del curso?"]:
    print("=" * 80)
    print("Usuario:", pregunta)
    print(asyncio.get_event_loop().run_until_complete(asistente_simulado(pregunta)))


## Paso 13: ejecución en Colab y local

### En Google Colab

Ejecuta las celdas de arriba hacia abajo. El notebook creará:

```text
/content/mcp_lab/curso_mcp.db
/content/mcp_lab/servidor_curso_mcp.py
```

Al reiniciar el runtime de Colab, esos archivos se pierden y el laboratorio vuelve a crearlos.

### En Jupyter local

Ejecuta el notebook en un entorno Python donde puedas instalar paquetes. El laboratorio creará:

```text
./mcp_lab/curso_mcp.db
./mcp_lab/servidor_curso_mcp.py
```

Para `stdio`, el cliente usa `sys.executable`, por lo que lanza el servidor con el mismo Python del notebook. Para Streamable HTTP, el notebook levanta el servidor como subproceso en `127.0.0.1` y se conecta a `/mcp`.


## Parte B: Streamable HTTP local

Ahora levantaremos el **mismo servidor MCP** usando Streamable HTTP. A diferencia de `stdio`, el servidor queda escuchando en un puerto local y el cliente se conecta a una URL.

En Colab esto funciona dentro del runtime:

```text
http://127.0.0.1:PUERTO/mcp
```

No es una URL pública. Sirve para aprender y probar el transporte HTTP desde el mismo notebook.


In [ ]:
def obtener_puerto_libre(host="127.0.0.1"):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((host, 0))
        return s.getsockname()[1]


def esperar_puerto(host: str, port: int, timeout: float = 15.0):
    inicio = time.time()
    ultimo_error = None
    while time.time() - inicio < timeout:
        try:
            with socket.create_connection((host, port), timeout=0.5):
                return True
        except OSError as e:
            ultimo_error = e
            time.sleep(0.2)
    raise TimeoutError(f"El puerto {host}:{port} no quedó disponible. Último error: {ultimo_error}")

HTTP_HOST = "127.0.0.1"
HTTP_PORT = obtener_puerto_libre(HTTP_HOST)
HTTP_URL = f"http://{HTTP_HOST}:{HTTP_PORT}/mcp"

http_env = os.environ.copy()
http_env["CURSO_MCP_DB"] = str(DB_PATH)

http_server = subprocess.Popen(
    [
        sys.executable,
        str(SERVER_PATH),
        "--transport",
        "streamable-http",
        "--host",
        HTTP_HOST,
        "--port",
        str(HTTP_PORT),
    ],
    env=http_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

try:
    esperar_puerto(HTTP_HOST, HTTP_PORT)
    print("Servidor Streamable HTTP listo:", HTTP_URL)
except Exception:
    salida = http_server.stderr.read() if http_server.stderr else ""
    raise RuntimeError(f"No se pudo iniciar el servidor HTTP. Logs:\n{salida[:3000]}")


## Paso 14: crear cliente MCP por Streamable HTTP

El cliente HTTP usa `streamable_http_client`. La sesión MCP es la misma idea que con `stdio`: inicializar, listar capacidades, leer resources, llamar tools y obtener prompts.


In [ ]:
from mcp.client.streamable_http import streamable_http_client


async def inspeccionar_servidor_http():
    async with streamable_http_client(HTTP_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            resources = await session.list_resources()
            prompts = await session.list_prompts()
            return {
                "tools": [t.name for t in tools.tools],
                "resources": [str(r.uri) for r in resources.resources],
                "prompts": [p.name for p in prompts.prompts],
            }

info_http = asyncio.get_event_loop().run_until_complete(inspeccionar_servidor_http())
print(json.dumps(info_http, ensure_ascii=False, indent=2))


## Paso 15: probar resources y tools por HTTP

El contrato del servidor no cambia. Lo único que cambia es el transporte.


In [ ]:
async def demo_http():
    async with streamable_http_client(HTTP_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            clase = await session.read_resource(AnyUrl("clase://4"))
            tareas = await session.call_tool("listar_tareas", {"estado": "pendiente"})
            progreso = await session.call_tool("resumen_progreso", {})
            return clase, tareas, progreso

clase_http, tareas_http, progreso_http = asyncio.get_event_loop().run_until_complete(demo_http())
print("RESOURCE clase://4")
print(clase_http.contents[0].text[:1000])
print("\nTOOL listar_tareas")
print(json.dumps(tareas_http.structuredContent, ensure_ascii=False, indent=2))
print("\nTOOL resumen_progreso")
print(json.dumps(progreso_http.structuredContent, ensure_ascii=False, indent=2))


## Paso 16: apagar el servidor HTTP

Como el servidor HTTP queda corriendo como subproceso, conviene apagarlo al terminar la demo. Si vuelves a ejecutar la sección HTTP, se elegirá otro puerto libre.


In [ ]:
if "http_server" in globals() and http_server.poll() is None:
    http_server.terminate()
    try:
        http_server.wait(timeout=5)
        print("Servidor HTTP detenido.")
    except subprocess.TimeoutExpired:
        http_server.kill()
        print("Servidor HTTP forzado a detenerse.")
else:
    print("No hay servidor HTTP activo.")


## Paso 17: seguridad del servidor construido

Buenas decisiones de este laboratorio:

- `stdio` no abre puertos de red.
- Streamable HTTP se limita a `127.0.0.1`.
- SQLite queda en `/content/mcp_lab` en Colab o `./mcp_lab` en local.
- En `stdio`, `stdout` queda reservado para JSON-RPC; los logs deben ir a `stderr`.
- Las tools de escritura exigen `confirmado=True`.
- Los resources solo entregan contexto y no modifican estado.
- Los errores de negocio usan `ToolError`.

Si expones Streamable HTTP fuera de `localhost`, agrega:

- TLS/HTTPS,
- autenticación y autorización,
- validación de `Origin`,
- rate limiting,
- auditoría persistente,
- permisos por tool,
- límites de tamaño para entradas y salidas,
- pruebas contra prompt injection.


## Ejercicios guiados

1. Agrega un resource `tareas://pendientes` y pruébalo por `stdio`.
2. Prueba el mismo resource por Streamable HTTP.
3. Agrega una tool `buscar_tareas(termino)`.
4. Modifica `crear_tarea` para rechazar títulos de menos de 8 caracteres.
5. Agrega un prompt `planificar_semana`.
6. Cambia `CURSO_MCP_DB` para apuntar a otra base SQLite local.
7. Compara qué cambia en el cliente entre `stdio` y Streamable HTTP.
8. Diseña una política de aprobación humana para tools que modifican datos.


## Preguntas finales

1. ¿Qué diferencia práctica hay entre `resources`, `tools` y `prompts`?
2. ¿Por qué `crear_tarea` y `marcar_tarea` exigen confirmación?
3. ¿Qué ventaja tiene `stdio` para Colab y ejecución local?
4. ¿Qué ventaja tiene Streamable HTTP frente a `stdio`?
5. ¿Qué problema aparece si el servidor imprime logs en `stdout` cuando usa `stdio`?
6. ¿Por qué conviene usar `sys.executable` al lanzar el servidor desde el notebook?
7. ¿Qué cambiarías antes de exponer Streamable HTTP fuera de `localhost`?
8. ¿Qué datos no deberías exponer como resource?


## Referencias

- PPT base: `Clase_9_Model_context_protocol.pdf`.
- MCP: https://modelcontextprotocol.io/docs/getting-started/intro
- Especificación MCP: https://modelcontextprotocol.io/specification
- SDK Python oficial: https://github.com/modelcontextprotocol/python-sdk
- Documentación del SDK Python: https://py.sdk.modelcontextprotocol.io/
- Construir servidores MCP con Python: https://py.sdk.modelcontextprotocol.io/server/
- Clientes MCP con Python: https://py.sdk.modelcontextprotocol.io/client/
- Streamable HTTP en FastMCP: https://py.sdk.modelcontextprotocol.io/server/#streamable-http-transport
